# Extrahiere Information aus dem Marc21

In [ ]:
class Book:
    def __init__(self, isbn, pub_date, language, pub_type):
        self.isbn = isbn
        self.pub_date = pub_date
        self.language = language
        self.pub_type = pub_type

    def __repr__(self):
        return f"Book(ISBN={self.isbn}, Publication Date={self.pub_date}, Language={self.language}, Publication Type={self.pub_type})"

    def __iter__(self):
        for i in range(0, 5):
            yield i

In [ ]:
book = Book(
    isbn="978-3-16-148410-0",
    pub_date="2023-10-01",
    language="English",
    pub_type="Hardcover"
)
for i in book:
    print(i)

In [ ]:
from lxml import etree

def extract_marc_info(xml):
    root = etree.parse(xml)

    ns = {"marc": "http://www.loc.gov/MARC21/slim"}

    # ISBN from 020$a
    isbn = root.xpath("marc:datafield[@tag='020']/marc:subfield[@code='a']/text()", namespaces=ns)
    isbn = isbn[0] if isbn else None

    # Publication date from 008 positions 7–10
    control_008 = root.xpath("marc:controlfield[@tag='008']/text()", namespaces=ns)
    pub_date = control_008[0][7:11] if control_008 else None

    # Language from 008 positions 35–37
    language = control_008[0][35:38] if control_008 else None

    # Publication type from Leader position 7
    leader = root.xpath("marc:leader/text()", namespaces=ns)
    pub_type_code = leader[0][6] if leader else None
    pub_type = {
        'a': 'Monograph',
        'b': 'Serial component part',
        'c': 'Collection',
        'd': 'Serial',
        'e': 'Reproduction',
        'f': 'Manuscript',
        'g': 'Archival Material',
        'h': 'Map',
        'i': 'Music',
        'j': 'Sound Recording',
        'k': 'Visual Material',
        'l': 'Software',
        'm': 'Mixed Materials',
        'n': 'Non-Material',
        'o': 'Other'
    }.get(pub_type_code, f"Unknown ({pub_type_code})")

    book = Book(isbn, pub_date, language, pub_type)

    return book

    
book = extract_marc_info("record.mrcx")
print(book)

# Decision Tree

Unsere Bedingungen für eine Aufnahme:
- Noch nicht im Bestand
- Muss in den letzten 5 Jahren erschienen sein
- Muss in deutsch, französisch oder englisch sein

Schliesslich soll noch bestimmt werden, ob die Spende an das Bücher- oder Zeitschriftenmagazin geht.

In [ ]:
books = []
books.append(extract_marc_info("record.mrcx"))
books.append(extract_marc_info("record.mrcx"))
books.append(extract_marc_info("record.mrcx"))
books.append(extract_marc_info("record.mrcx"))


In [ ]:
books[1].isbn = "978-3-16-148410-0"
books[2].isbn = "978-3-16-148410-2"
books[3].isbn = "978-3-16-148410-2"

In [ ]:
library = ["978-3-16-148410-0", "978-1-234-56789-7", "978-0-123456-47-2"]
CUTOFF = 2020
ALLOWED_LANGUAGES = ["ger", "fre", "eng"]

# für jedes Buch (book) in der Büchersammlung (books)
for i in range(2, 5):
    book = books[i]
    print(f"Processing book {i + 1}: {book}")
    if book.isbn in library:
        print("Book already exists in the library.")
        continue
    if int(book.pub_date) < 2020:
        print("Book published before cutoff date.")
        continue
    if book.language not in ALLOWED_LANGUAGES:
        print("Book language not allowed.")
        continue

    if book.pub_type in ["Monograph", "Collection"]:
        library.append(book.isbn)
        print("Add book to Büchermagazin.")
    elif book.pub_type == "Serial":
        library.append(book.isbn)
        print("Add book to Zeitschriftenmagazin.")
    else:
        print("Book is not a valid publication type.")
else:
    print("All books processed.")

In [ ]:
library = ["978-3-16-148410-0", "978-1-234-56789-7", "978-0-123456-47-2"]
CUTOFF = 2020
ALLOWED_LANGUAGES = ["ger", "fre", "eng"]

if book.isbn not in library:
    if int(book.pub_date) >= 2020:
        if book.language in ALLOWED_LANGUAGES:
            if book.pub_type in ["Monograph", "Collection"]:
                library.append(book.isbn)
                print("Book added to the library.")
            elif book.pub_type == "Serial":
                print("Add book to Zeitschriftenmagazin.")
            else:
                print("Book is not a valid publication type.")
        else:
            print("Book is not in an allowed language.")
    else:
        print("Book is too old.")
else:
    print("Book already exists in the library.")

In [ ]:
if (book.isbn not in library) and \
    (int(book.pub_date) >= 2020) and \
        book.language in ALLOWED_LANGUAGES:
    if book.pub_type in ["Monograph", "Collection"]:
        library.append(book.isbn)
        print("Add book to Büchermagazin.")
    elif book.pub_type == "Serial":
        library.append(book.isbn)
        print("Add book to Zeitschriftenmagazin.")
    else:
        print("Book is not a valid publication type.")
else:
    print("Book not added to the library.")